In [55]:
import pandas as pd
import numpy as np


In [56]:
day="3rd August"
data=pd.read_csv('aug3.csv')
data.head()

,Timestamp,Name,Roll No,Distance
0,8/3/2026 16:02:24,test,test,76876 meters away
1,8/3/2026 17:46:02,Test,0000,14 meters away
2,8/3/2026 19:45:20,Jeny Prafulbhai Hinsu,Im26026,14 meters away
3,8/3/2026 19:45:23,Ankit Kumar,B26007,9 meters away
4,8/3/2026 19:45:24,Roman M,B26229,10 meters away


In [57]:
output=pd.read_csv("complete_attendance.csv")
output.head()

,Roll Number,3rd August,4th August,5th August,6th August,7th August,8th August,9th August
0,B26001,0,0,0,0,0,0,0
1,B26002,0,0,0,0,0,0,0
2,B26003,0,0,0,0,0,0,0
3,B26004,0,0,0,0,0,0,0
4,B26005,0,0,0,0,0,0,0


In [58]:
roll=data['Roll No'].tolist()
for i in range(len(roll)):
    roll[i]=roll[i].strip()
    roll[i]=roll[i].upper()
print(roll)
    
    

['TEST', '0000', 'IM26026', 'B26007', 'B26229', 'B26435', 'B26542', 'B26480', 'B26553', 'B26208', 'B26156', 'B26621', 'B26467', 'B26407', 'B26137', 'B26565', 'B26444', 'B26130', 'B26176', 'B26041', 'B26309', 'B26175', 'B26531', 'B26192', 'B26545', 'B26567', 'B26568', 'B26162', 'B26544', 'B26228', 'B26353', 'B26008', 'B26015', 'B26100', 'B26069', 'B26551', 'B26180', 'B26172', 'B26592', 'B26194', 'B26280', 'B26110', 'B26555', 'B26406', 'IM26008', 'B26404', 'B26543', 'B26520', 'B26170', 'B26583', 'B26217', 'B26267', 'B26375', 'B26258', 'B26518', 'B26076', 'B26508', 'B26250', 'B26052', 'B26348', 'B26501', 'B26227', 'B26286', 'B26326', 'B26313', 'B26330', 'B26511', 'B26619', 'B26475', 'B26350', 'B26179', 'B26562', 'B26033', 'B26393', 'B26266', 'B26210', 'B26291', 'IM26071', 'B26174', 'B26235', 'B26071', 'B26574', 'B26486', 'B26471', 'B26243', 'B26378', 'B26595', 'B26368', 'B26578', 'B26522', 'B26149', 'B26273', 'B26113', 'B26296', 'B26546', 'B26364', 'B26452', 'B26337', 'B26028', 'B26010', 

In [59]:
def generate_attendance():
    for i in roll:
        output.loc[output['Roll Number']==i,day]=1
    print(output.head(100))
    output.to_csv("complete_attendance_test2.csv",index=False)   
        

In [62]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

def analyze_and_plot_attendance(file_path, day='3rd August'):
    # 1. Load Data
    df = pd.read_csv(file_path)

    # 2. Categorize Program based on Roll Number prefix
    def get_program(roll):
        roll_str = str(roll).strip()
        if roll_str.startswith('B26'):
            return 'BTech'
        elif roll_str.startswith('IM26'):
            return 'IMBA'
        else:
            return 'Other'

    df['Program'] = df['Roll Number'].apply(get_program)
    date_cols = [c for c in df.columns if c not in ['Roll Number', 'Program']]

    # Check if specified day exists in the dataset
    if day not in date_cols:
        print(f"Error: '{day}' was not found in the dataset dates.")
        print(f"Available dates: {date_cols}")
        return

    # 3. Reshape Data (Long format)
    df_long = df.melt(
        id_vars=['Roll Number', 'Program'], 
        value_vars=date_cols, 
        var_name='Date', 
        value_name='Status'
    )
    
    # Normalize status labels
    df_long['Status'] = df_long['Status'].astype(str).str.strip().str.capitalize()
    status_map = {'1': 'Present', '0': 'Absent', 'Proxy': 'Proxy'}
    df_long['Status'] = df_long['Status'].map(status_map).fillna(df_long['Status'])

    # Filter data for the requested day
    selected_day_df = df_long[df_long['Date'] == day]

    # --- PRINT SUMMARY STATISTICS ---
    print("=" * 60)
    print(f" SUMMARY STATISTICS FOR {day.upper()}")
    print("=" * 60)
    
    print("\n1. TOTAL STUDENT COUNT BY PROGRAM:")
    print(df['Program'].value_counts().to_string())

    print(f"\n2. ATTENDANCE SUMMARY FOR {day}:")
    day_summary = selected_day_df.groupby(['Program', 'Status']).size().unstack(fill_value=0)
    print(day_summary.to_string())

    print(f"\n3. PROXY INCIDENTS DETECTED ON {day}:")
    proxies = selected_day_df[selected_day_df['Status'] == 'Proxy']
    if not proxies.empty:
        print(proxies[['Roll Number', 'Program', 'Date']].to_string(index=False))
    else:
        print(f"No proxy attendance found on {day}.")
    print("=" * 60)

    # File naming helper (converts "3rd August" -> "3rd_august")
    safe_day_str = day.lower().replace(' ', '_')

    # --- GENERATE & SAVE GRAPHS ---

    # Graph 1: Program Distribution (Pie Chart)
    plt.figure(figsize=(6, 6))
    program_counts = df['Program'].value_counts()
    plt.pie(
        program_counts, 
        labels=program_counts.index, 
        autopct='%1.1f%%', 
        colors=['#4C72B0', '#DD8452'], 
        startangle=140, 
        explode=(0.05, 0)
    )
    plt.title('Student Distribution by Program', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('program_distribution.png', dpi=300)
    plt.close()
    print(" Saved: program_distribution.png")

    # Graph 2: Attendance Comparison on Specified Day (Bar Chart)
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(
        data=selected_day_df, 
        x='Program', 
        hue='Status', 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title(f'Attendance Status Comparison ({day})', fontsize=14, fontweight='bold')
    plt.xlabel('Program', fontsize=12)
    plt.ylabel('Number of Students', fontsize=12)
    
    # Add data values on top of bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
    
    plt.tight_layout()
    day_bar_filename = f'{safe_day_str}_attendance.png'
    plt.savefig(day_bar_filename, dpi=300)
    plt.close()
    print(f" Saved: {day_bar_filename}")

    # Graph 3: Date-wise Attendance Trend (Line Chart across all days)
    plt.figure(figsize=(10, 5))
    trend_df = df_long.groupby(['Date', 'Status']).size().reset_index(name='Count')
    sns.lineplot(
        data=trend_df, 
        x='Date', 
        y='Count', 
        hue='Status', 
        marker='o', 
        linewidth=2.5, 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title('Overall Attendance Trend Across All Dates', fontsize=14, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Total Count', fontsize=12)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig('attendance_trend.png', dpi=300)
    plt.close()
    print(" Saved: attendance_trend.png")

# Run analysis for a specific day
if __name__ == "__main__":
    # Change 'day' variable to any date present in your dataset 
    # Available options: '3rd August', '4th August', '5th August', '6th August', '7th August', '8th August', '9th August'
    target_day = "3rd August"
    
    analyze_and_plot_attendance('complete_attendance_v0.csv', day=target_day)

 SUMMARY STATISTICS FOR 3RD AUGUST

1. TOTAL STUDENT COUNT BY PROGRAM:
Program
BTech    640
IMBA      80

2. ATTENDANCE SUMMARY FOR 3rd August:
Status   Absent  Present  Proxy
Program                        
BTech        73      565      2
IMBA         63       17      0

3. PROXY INCIDENTS DETECTED ON 3rd August:
Roll Number Program       Date
     B26382   BTech 3rd August
     B26499   BTech 3rd August
 Saved: program_distribution.png
 Saved: 3rd_august_attendance.png
 Saved: attendance_trend.png
